# 🌿 Demeter SDK — Demo en Google Colab

**Servidor:** `https://patata.monters.org`  
**API docs:** `https://patata.monters.org/api/docs`

Para obtener tu API Key: entra en `https://patata.monters.org`, crea un experimento y copia su clave desde el panel LIMS.

In [ ]:
# ─── Celda 1: Instalación ─────────────────────────────────────────────────────
!pip install --index-url https://test.pypi.org/simple/ \
             --extra-index-url https://pypi.org/simple/ \
             demeter_sdk -q

import demeter_sdk
print(f'Demeter SDK v{demeter_sdk.__version__} ✅')

In [ ]:
# ─── Celda 2: Configuración ───────────────────────────────────────────────────
# El servidor de producción — Cloudflare Tunnel + Traefik
BASE_URL = 'https://patata.monters.org'

# API Key del experimento en la DB.
# Cómo obtenerla: en el servidor ejecuta:
#   docker exec demeter-db psql -U postgres -d demeter_db \
#     -c "SELECT name, api_key FROM experimentos;"
API_KEY = 'PEGA_AQUI_TU_API_KEY'

EXP_ID        = 1         # ID del experimento a analizar
DIAS          = 30        # días de histórico
FECHA_SIEMBRA = '2025-09-01'

In [ ]:
# ─── Celda 3: Crear cliente y verificar conexión ──────────────────────────────
from demeter_sdk import DemeterClient

client = DemeterClient(api_key=API_KEY, base_url=BASE_URL)

ok = client.ping()
print(f'Server: {"✅ Online" if ok else "❌ Offline"} → {BASE_URL}')

if not ok:
    raise RuntimeError('Servidor no accesible. Comprueba que el stack está levantado en patata.monters.org')

In [ ]:
# ─── Celda 4a: Descargar y enriquecer datos reales ───────────────────────────
df = client.get_enriched_data(
    experimento_id=EXP_ID,
    dias=DIAS,
    fecha_siembra=FECHA_SIEMBRA,
)

print(f'📦 {len(df)} registros | {df["node_id"].nunique()} nodos')
df[['timestamp', 'temperature', 'humidity', 'vpd_kpa', 'dew_point_c', 'dap_days']].head(10)

In [ ]:
# ─── Celda 4b (ALTERNATIVA): Datos offline si el servidor no está disponible ──
# Descomenta y ejecuta si no tienes acceso al servidor ahora mismo

# import numpy as np
# from datetime import datetime, timedelta
# from demeter_sdk import transform, science
#
# base = datetime(2025, 10, 1)
# raw = [{
#     'timestamp': (base + timedelta(hours=i)).isoformat(),
#     'temperature': round(22 + 6 * np.sin(i / 10), 2),
#     'humidity':    round(72 + 12 * np.cos(i / 8), 2),
#     'node_id':     (i % 3) + 1,
# } for i in range(720)]  # 30 días x 24h
#
# df = science.enrich(transform.pipeline(raw), fecha_siembra='2025-09-01')
# print(f'Demo offline: {len(df)} registros generados')

In [ ]:
# ─── Celda 5: Estadísticas resumen ───────────────────────────────────────────
cols = ['temperature', 'humidity', 'vpd_kpa', 'dew_point_c']
df[cols].describe().round(3)

In [ ]:
# ─── Celda 6: VPD con zonas de estrés ────────────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
from demeter_sdk import viz

fig = viz.plot_vpd(df)
plt.show()

In [ ]:
# ─── Celda 7: Timeseries de temperatura ──────────────────────────────────────
fig = viz.plot_timeseries(df, sensor='temperature', moving_avg=True)
plt.show()

In [ ]:
# ─── Celda 8: Boxplot por nodo ────────────────────────────────────────────────
fig = viz.plot_boxplot(df, metric='vpd_kpa')
plt.show()

In [ ]:
# ─── Celda 9: Heatmap temperatura × nodo ─────────────────────────────────────
fig = viz.plot_heatmap(df, metric='temperature', resample_rule='1D')
plt.show()

In [ ]:
# ─── Celda 10: Exportar Excel y descargar ────────────────────────────────────
from demeter_sdk import export

path = export.to_excel(
    df_enriched=df,
    path='/tmp/demeter_patata_report.xlsx',
    experiment_name=f'Experimento {EXP_ID} — patata.monters.org',
)
print(f'Excel generado: {path} ({path.stat().st_size // 1024} KB)')

try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    print(f'Archivo disponible en: {path}')

In [ ]:
# ─── Celda 11: Exportar CSV ───────────────────────────────────────────────────
csv_path = export.to_csv(df, '/tmp/telemetria_patata.csv')
print(f'CSV: {csv_path} ({csv_path.stat().st_size // 1024} KB)')

try:
    from google.colab import files
    files.download(str(csv_path))
except ImportError:
    pass

---
## Obtener la API Key de tu servidor

Ejecuta esto en el servidor (SSH) para ver las API keys de los experimentos creados por el seeder:

```bash
docker exec demeter-db psql -U postgres -d demeter_db \
  -c "SELECT id, name, api_key FROM experimentos ORDER BY id;"
```

Copia el valor de `api_key` del experimento que quieras analizar y pégalo en la **Celda 2** arriba.